# EU Gisco file from 2024 https://ec.europa.eu/eurostat/web/gisco/geodata/administrative-units/countries

In [10]:
import geopandas as gpd 
gdf = gpd.read_file("data/CNTR_RG_01M_2024_4326.gpkg")
gdf.to_parquet("data/CNTR_RG_01M_2024_4326.parquet")

# Boundary file from https://sashamaps.net/docs/resources/europe-asia-boundary/

Note that the geometries are accurate from the file quality is terrible, so it needs to be fixed. Finding these errors cost me an entire evening as visually everything looked fine.

In [11]:
import geopandas as gpd 
gdf = gpd.read_file("data/asia_europe_border.geojson")
gdf

,geometry
0,"LINESTRING (25.68963 39.88002, 25.6891 39.8788..."
1,"LINESTRING (25.68963 39.88002, 25.86068 39.983..."
2,"LINESTRING (35.50694 43.43556, 34.21972 43.188..."
3,"LINESTRING (29.82667 42.49, 29.975 42.5575, 30..."
4,"LINESTRING (44.82914 42.6178, 44.83494 42.6188..."
...,...
459,"LINESTRING (67.47084 68.29232, 67.58216 68.288..."
460,"LINESTRING (67.47084 68.29232, 67.33723 68.285..."
461,"LINESTRING (38.24825 44.6076, 38.25753 44.6086..."
462,"LINESTRING (38.24825 44.6076, 38.23961 44.6112..."


In [12]:
import geopandas as gpd
from shapely.ops import linemerge
from shapely.geometry import LineString, MultiLineString

def create_contiguous_line(input_file, output_file):
    # 1. Load the "terrible" GeoJSON
    gdf = gpd.read_file(input_file)
    
    # 2. Attempt to merge segments
    # linemerge handles direction flipping and reordering automatically
    merged_geometry = linemerge(gdf.geometry.tolist())
    
    # 3. Handle cases where linemerge still results in multiple parts (gaps)
    if merged_geometry.geom_type == 'MultiLineString':
        print(f"Warning: Gaps found. Merging {len(merged_geometry.geoms)} segments into one...")
        
        # Extract individual lines from the MultiLineString
        parts = list(merged_geometry.geoms)
        combined_coords = []
        
        for i, part in enumerate(parts):
            curr_coords = list(part.coords)
            if not combined_coords:
                combined_coords.extend(curr_coords)
            else:
                # Check if we should flip the part to match the last point
                # (Though linemerge usually handles this, we do it here for manual gaps)
                last_point = combined_coords[-1]
                first_of_part = curr_coords[0]
                last_of_part = curr_coords[-1]
                
                # Simple distance check to find which end of the new part is closer
                dist_to_start = ((last_point[0]-first_of_part[0])**2 + (last_point[1]-first_of_part[1])**2)**0.5
                dist_to_end = ((last_point[0]-last_of_part[0])**2 + (last_point[1]-last_of_part[1])**2)**0.5
                
                if dist_to_end < dist_to_start:
                    curr_coords.reverse()
                
                combined_coords.extend(curr_coords)
        
        final_geometry = LineString(combined_coords)
    else:
        final_geometry = merged_geometry

    # 4. Create a new GeoDataFrame with just this one line
    final_gdf = gpd.GeoDataFrame(
        {'name': ['asia_europe_border']}, 
        geometry=[final_geometry], 
        crs=gdf.crs
    )

    # 5. Save as GeoParquet
    # Note: requires 'pyarrow' installed: pip install pyarrow
    final_gdf.to_parquet(output_file)
    print(f"Success! Contiguous line saved to {output_file}")

# Usage
create_contiguous_line("data/asia_europe_border.geojson", "data/asia_europe_border_fixed.parquet")

Success! Contiguous line saved to data/asia_europe_border_fixed.parquet


# Now let's create a polygon for easier clipping (as most programs out there don't support clipping a polygon by line (QGIS not, Saga does))

In [13]:
import geopandas as gpd
from shapely.geometry import Polygon

def line_to_polygon(input_parquet, output_parquet):
    # 1. Load the contiguous line from the Parquet file
    gdf = gpd.read_parquet(input_parquet)
    
    # Ensure we are working with the single geometry we created earlier
    line = gdf.geometry.iloc[0]
    
    if line.geom_type != 'LineString':
        raise ValueError(f"Expected LineString, found {line.geom_type}")

    # 2. Convert LineString to Polygon
    # Shapely's Polygon constructor automatically closes the ring by 
    # connecting the last point back to the first point in a straight line.
    poly = Polygon(line)

    # 3. Validation (Fixes self-intersections if the border line crosses itself)
    if not poly.is_valid:
        print("Geometry is self-intersecting. Fixing...")
        poly = poly.buffer(0) 

    # 4. Create new GeoDataFrame
    poly_gdf = gpd.GeoDataFrame(
        {'name': ['asia_europe_polygon_area']}, 
        geometry=[poly], 
        crs=gdf.crs
    )

    # 5. Save as GeoParquet
    poly_gdf.to_parquet(output_parquet)
    print(f"Success! Polygon saved to {output_parquet}")

# Usage
line_to_polygon("data/asia_europe_border_fixed.parquet", "data/asia_europe_polygon.parquet")

Geometry is self-intersecting. Fixing...
Success! Polygon saved to data/asia_europe_polygon.parquet


# Let's use the GISCO file 

In [14]:
import duckdb

# 1. Setup Data
europe_iso3_extended = [
    "ALB", "AND", "AUT", "BEL", "BGR", "BIH", "BLR", "CHE", "CYP", "CZE", 
    "DEU", "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "HRV", "HUN", 
    "IRL", "ISL", "ITA", "LIE", "LTU", "LUX", "LVA", "MCO", "MDA", "MKD", 
    "MLT", "MNE", "NLD", "NOR", "POL", "PRT", "ROU",
    "RUS",
    "SMR", "SRB", 
    "SVK", "SVN", "SWE", "TUR", "UKR", "VAT", "XKX", "FRO", "ALA", 
    "SJM", "IMN", "JEY", "GGY", "GIB"
]
output_file = "data/europe.parquet"

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

export_query = f"""
    COPY (
        SELECT 
            *
        FROM read_parquet('data/CNTR_RG_01M_2024_4326.parquet')
        WHERE ISO3_CODE IN $1
    ) TO '{output_file}' (FORMAT PARQUET);
"""

# 4. Execute
con.execute(export_query, [europe_iso3_extended])

print(f"Success! Exported to {output_file}")

Success! Exported to data/europe.parquet


# Clip everything and remove anything East of Novosibirsk

In [16]:
import geopandas as gpd
from shapely.geometry import box

def process_european_continent():
    # 1. Load the input data
    europe = gpd.read_parquet('data/europe.parquet')
    asia_overlay = gpd.read_parquet('data/asia_europe_polygon.parquet')

    # Ensure EPSG:4326 (WGS84) for longitude-based clipping
    if europe.crs != "EPSG:4326":
        europe = europe.to_crs("EPSG:4326")
    if asia_overlay.crs != "EPSG:4326":
        asia_overlay = asia_overlay.to_crs("EPSG:4326")

    # 2. Subtract the Asian continent polygon
    europe_refined = gpd.overlay(europe, asia_overlay, how='difference')

    # 3. Fix the "Far East" issue: 
    # Use a bounding box that starts West of Iceland (-30) and ends at Novosibirsk (82.93).
    # This prevents the script from picking up Russian territory that wraps past the 180 line.
    west_limit = -30.0 
    novosibirsk_lon = 82.93
    
    # box(minx, miny, maxx, maxy)
    # We use -30 to -90 to 82.93 to 90
    selection_box = box(west_limit, -90, novosibirsk_lon, 90)
    
    # Clip to the specific European/Western Russian window
    european_continent = europe_refined.clip(selection_box)

    # 4. Save the result
    european_continent.to_parquet('european_continent.parquet')
    european_continent.to_file('european_continent.geojson')
    print("Successfully saved european_continent.parquet")

process_european_continent()

Successfully saved european_continent.parquet
